## hiya!

I've written this notebook to understand Hugging Face's `transformers`, how to use their pre-trained models and overall basics of fine-tuning.

This is an altered version of [Ertuğrul Demir's notebook](https://www.kaggle.com/code/datafan07/disaster-tweets-nlp-eda-bert-with-transformers#Predicting-and-Submission) (don't forget to check them out!) excluding the Exploratory Data Analysis (which I might add soon!), focusing only on training and prediction. It's fully rewritten by hand with quite a bit of changes, some of which are variable names and function structures.

Notes:
- Data is considered labeled if labels are assigned to it by default. The data in the training CSV file is considered labeled, since labels are given for each feature. After loading and preprocessing, dividing the "training" data into data for training (training once again!) and evaluation would be kind of contradictory.

Hope you enjoy ;)

# Initializing

<!-- We'll need the following libraries:
- torch (for data pipelines and training/evaluating the model)
- transformers (mainly for the tokenizer and model classes)
- numpy (processing data from arrays)
- pandas (reading input data)
- matplotlib (data visualization)
- sklearn (its `metrics` module to evaluate the model's predictions)
- time, datetime (calculating how much time specific processes took) -->

We'll need quite a lot of libraries:

- `torch` and `transformers` will be used to load and train the model,
- `numpy`, `pandas` and `torch.utils.data` will be used to load and preprocess the data,
-  `matplotlib` will be used to visualize training and evaluation losses on each epoch,
- `sklearn.metrics` to calculate metrics to better understand the model's progress,
- and finally, `time` and `datetime` are gonna be utilized to compute how much time specific processes took.

In [ ]:
#! machine learning
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

#! data related
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader, random_split, RandomSampler, SequentialSampler

#! metrics calculation
from sklearn.metrics import accuracy_score, f1_score

#! runtime calculation
import time
import datetime

The variable below will control the model variant used. Possible options are:
- `bert-large-uncased`
- `bert-base-uncased`

Obviously, the larger the model, the better patterns it'll learn. Only in this case though, usually because that's not how it's done on practice! Larger models require a lot of hyperparameter tuning (for example, finding the best dimensions for every layer and picking the best learning rate for the most suitable optimizer), discovering new techniques, architectures and performing other perfection rituals, which is the reason why releasing newer and better State-Of-The-Art models takes so long.

TL;DR it's better to use the larger model, though only in this case!

In [ ]:
model_name = "bert-large-uncased"

# Loading and processing data #

Our BERT model will take encoded sentences and output an object containing logits and losses. The said sentences are encoded using the respective model's tokenizer. We can use it by passing the model's name into `AutoTokenizer.from_pretrained()` from the Hugging Face's `tokenizers` library.

Before doing that, we'll fetch our data. Training data, as usual, has two main columns - `text` and `target` which respectively contain the features and the targets. We'll only need the `text` column from the test data.

Let's look at the files we've got:

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

These directories will be stored in the respective variables for convenience.

In [ ]:
base_dir = "/kaggle/input/nlp-getting-started/"
train_dir = base_dir + "train.csv"
test_dir = base_dir + "test.csv"
sample_submission_dir = base_dir + "sample_submission.csv"

Let's load the dataframes...

In [ ]:
train_dataframe = pd.read_csv(train_dir)
test_dataframe = pd.read_csv(test_dir)

train_dataframe.sample(5)

...and combine them to find the longest encoded sentence (I'll explain why below)

In [ ]:
combined_dataframe = pd.concat([train_dataframe, test_dataframe])
combined_data = combined_dataframe.text.to_numpy()

Here's the tokenizer definition. Pretty easy, right?

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

And an example of it's usage:

In [ ]:
print(f"Original sentence: {combined_data[0]}")

print(f"Tokenized sentence: {tokenizer.tokenize(combined_data[0])}")

print(f"Encoded sentence: {tokenizer.convert_tokens_to_ids(tokenizer.tokenize(combined_data[0]))}")

The model requires the encoded sentences to be of the same length (meaning the tensors containing them must be of the same length). The cell below will process the data and find the longest encoded sentence. Its length will be used to pad the smaller tensors.

In [ ]:
max_sentence_length = 0

for sentence in combined_data:
    encoded = tokenizer.encode(sentence, add_special_tokens=True)
    max_sentence_length = max(max_sentence_length, len(encoded))

print(f"Maximum sentence length is {max_sentence_length} IDs long.")

The `add_special_tokens` parameter defines whether the tokenizer has to add *special tokens* to the output. Those tokens are native to BERT, which include `[CLS]` (start of the sentence), `[SEP]` (end of a sentence), `[UNK]` (represents a word which the tokenizer doesn't recognize) and so on. Adding them here is necessary since they elongate the encoded sentences.

Simply using `tokenizer.encode` won't let us specify the important details, for that reason we'll use its cooler sibling - `tokenizer.encode_plus`.

If the model would return attention masks (which mark tokens that aren't pads), it'd take much more time. Preloading it along with the input IDs (encoded sentences) from the tokenizer would be much computationally efficient.

In [ ]:
def preprocess_text(data, targets=None):
    input_ids = []
    attention_masks = []
    
    for sentence in data:
        encoded_dict = tokenizer.encode_plus(
            sentence,
            add_special_tokens=True,
            max_length=max_sentence_length,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt"
        )

        input_ids.append(encoded_dict["input_ids"])
        attention_masks.append(encoded_dict["attention_mask"])

    input_ids = torch.cat(input_ids)
    attention_masks = torch.cat(attention_masks)

    if targets is not None: # if targets were passed
        targets = torch.tensor(targets)
        return input_ids, attention_masks, targets
    else:
        return input_ids, attention_masks

The function logic is pretty simple. If you pass the targets, the function will return their tensor version. Otherwise the targets remain as `None` and you only recieve input IDs and attention masks.

In [ ]:
labeled_input_ids, labeled_attention_masks, targets = preprocess_text(
    data=train_dataframe.text.to_numpy(),
    targets=train_dataframe.target.to_numpy()
)

test_input_ids, test_attention_masks = preprocess_text(data=test_dataframe.text.to_numpy())

In [ ]:
print(f"Input IDs' shape is {labeled_input_ids.shape}\n")

print(f"Attention masks' shape is {labeled_attention_masks.shape}\n")

print(f"Targets' shape is {targets.shape}\n")

Datasets construction is pretty simple. Just pass your tensor data into the `TensorDataset` class and you get a PyTorch `Dataset` instance.

We'll also split the labeled data into training and evaluation datasets. It's important to see how well the model performs on the data it's never seen before.

In [ ]:
labeled_dataset = TensorDataset(labeled_input_ids, labeled_attention_masks, targets)

train_dataset_size = int(0.8 * len(labeled_dataset))
validation_dataset_size = len(labeled_dataset) - train_dataset_size

train_dataset, validation_dataset = random_split(labeled_dataset, [train_dataset_size, validation_dataset_size])

print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(validation_dataset)}")

The testing dataset initialization isn't as flamboyant as the ones above.

In [ ]:
test_dataset = TensorDataset(test_input_ids, test_attention_masks)

print(f"Test dataset size: {len(test_dataset)}")

Creating dataloaders isn't that hard as well - you need to specify the dataset to fetch data from, the batch size to use and the sampler containing the said dataset.

Samplers define how data is fetched from datasets. `RandomSampler` specifies that the data is fetched at random and `SequentialSampler` means that the data is fetched sequentially (i. e. one batch after another - like a `for` loop). There are more but they're a completely different story.

In [ ]:
batch_size = 32

train_sampler = RandomSampler(train_dataset)
train_dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    sampler=train_sampler
)

validation_sampler = SequentialSampler(validation_dataset)
validation_dataloader = DataLoader(
    validation_dataset,
    batch_size=batch_size,
    sampler=validation_sampler
)

test_sampler = SequentialSampler(test_dataset)
test_dataloader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    sampler=test_sampler
)

next(iter(train_dataset))[0].shape

# Assembling and training the model #

Fine-tuning models takes a lot of time, especially with larger versions. Thankfully, Kaggle offers us 30 hours per week of accelerator (GPU/TPU) usage. Let's check if we have a CUDA accelerator, in which case we'll use it as our device.

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"> Using {torch.cuda.get_device_name(0)}")
    print(f"> Number of GPUs: {torch.cuda.device_count()}")
else:
    device = torch.device("cpu")
    print(f"> Using CPU")

The model definition is fairly simple. We'll tell the model loading class not to return hidden states (we don't need them at all) and attention masks (since we've already loaded them into memory).

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    output_attentions=False,
    output_hidden_states=False)

model.to(device)

You can experiment with different optimizers. I recommend using `AdamW`, but it's fully up to you to decide!

In [ ]:
optimizer = torch.optim.AdamW(
    params=model.parameters(),
    lr=6e-6,
    eps=1e-8
)

epochs = 3
total_train_steps = len(train_dataloader) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_train_steps
)

scheduler

The names of the functions below are self-explanatory. `calculate_batch_accuracy` calculates how accurate the model is at prediction (using batches of data), `calculate_batch_f1_score` calculates the F1 score and `format_time` transforms the given elapsed time (in seconds) to a string of format HH:MM:SS.

In [ ]:
def calculate_batch_accuracy(logits, targets):
    flattened_predictions = np.argmax(logits, axis=-1).flatten()
    flattened_targets = targets.flatten()
    
    return accuracy_score(
        y_true=flattened_targets,
        y_pred=flattened_predictions)

def calculate_batch_f1_score(logits, targets):
    flattened_predictions = np.argmax(logits, axis=-1).flatten()
    flattened_targets = targets.flatten()
    
    return f1_score(
        y_true=flattened_targets,
        y_pred=flattened_predictions)

def format_time(elapsed_seconds: time.time) -> str:
    return str(
        datetime.timedelta(seconds=round(elapsed_seconds))
    )

The most important part - training the model. It can take a while - as at the time of writing, the model took 8 minutes to train, even with the P100 GPU.

We'll also store the model statistics (metrics, respective epochs, etc.) to visualize later.

In [ ]:
training_stats: list[dict] = []
train_start_time = time.time()

for epoch in range(epochs):
    print(f"=========== Epoch {epoch+1} ===========")
    
    # training epoch
    print("Training...")

    # zero out the stats to track
    train_epoch_start_time = time.time()
    total_train_loss = 0.0
    total_train_accuracy = 0.0
    total_train_f1_score = 0.0

    # the training loop
    model.train()
    for batch_input_ids, batch_attention_masks, batch_targets in train_dataloader:
        # zero out the gradients
        optimizer.zero_grad()

        # put the data on the found device
        batch_input_ids = batch_input_ids.to(device)
        batch_attention_masks = batch_attention_masks.to(device)
        batch_targets = batch_targets.to(device)

        # fetch the model's predictions
        outputs = model(
            batch_input_ids,
            token_type_ids=None,
            attention_mask=batch_attention_masks,
            labels=batch_targets
        )
        
        logits = outputs.logits
        loss = outputs.loss

        # put the data to calculate metrics with on the cpu
        logits = logits.detach().cpu()
        batch_targets = batch_targets.cpu()

        # add the stats to the respective variables
        total_train_loss += loss.item()
        total_train_accuracy += calculate_batch_accuracy(logits, batch_targets)
        total_train_f1_score += calculate_batch_f1_score(logits, batch_targets)

        # backward, clip gradients, optimizer and scheduler steps
        # clipping the gradients prevents them from exploding
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

    # calculate the average stats
    average_train_loss = total_train_loss / len(train_dataloader)
    average_train_accuracy = total_train_accuracy / len(train_dataloader)
    average_train_f1_score = total_train_f1_score / len(train_dataloader)
    train_epoch_elapsed = format_time(time.time() - train_epoch_start_time)

    # print the results
    print(f"   > Average loss: {average_train_loss:.4f}")
    print(f"   > Average accuracy: {average_train_accuracy:.4f}")
    print(f"   > Average f1 score: {average_train_f1_score:.4f}")
    print(f"   > Elapsed: {train_epoch_elapsed}\n")

    # evaluating epoch
    print("Evaluating...")

    # zero out the stats to track
    validation_epoch_start_time = time.time()
    total_validation_loss = 0.0
    total_validation_accuracy = 0.0
    total_validation_f1_score = 0.0

    # evaluating loop
    model.eval()
    for batch_input_ids, batch_attention_masks, batch_targets in validation_dataloader:
        # put the data on the used device
        batch_input_ids = batch_input_ids.to(device)
        batch_attention_masks = batch_attention_masks.to(device)
        batch_targets = batch_targets.to(device)

        # fetch the outputs in inference mode to stop the gradients
        with torch.inference_mode():
            outputs = model(
                batch_input_ids,
                token_type_ids=None,
                attention_mask=batch_attention_masks,
                labels=batch_targets
            )
            
            logits = outputs.logits
            loss = outputs.loss

        # put the data to calculate metrics with on the cpu
        logits = logits.detach().cpu()
        batch_targets = batch_targets.cpu()

        # add the calculated metrics to their cumulative variables
        total_validation_loss += loss.item()
        total_validation_accuracy += calculate_batch_accuracy(logits, batch_targets)
        total_validation_f1_score += calculate_batch_f1_score(logits, batch_targets)

    # calculate the average stats
    average_validation_loss = total_validation_loss / len(validation_dataloader)
    average_validation_accuracy = total_validation_accuracy / len(validation_dataloader)
    average_validation_f1_score = total_validation_f1_score / len(validation_dataloader)
    validation_epoch_elapsed = format_time(time.time() - validation_epoch_start_time)

    # print what's going on
    print(f"   > Average loss: {average_validation_loss:.4f}")
    print(f"   > Average accuracy: {average_validation_accuracy:.4f}")
    print(f"   > Average f1 score: {average_validation_f1_score:.4f}")
    print(f"   > Elapsed: {validation_epoch_elapsed}\n")

    # add the epoch results to the stats tracking array
    training_stats.append({
        "epoch": epoch + 1,
        "train loss": average_train_loss,
        "train accuracy": average_train_accuracy,
        "train f1 score": average_train_f1_score,
        "train time": train_epoch_elapsed,
        "validation loss": average_validation_loss,
        "validation accuracy": average_validation_accuracy,
        "validation f1 score": average_validation_f1_score,
        "validation time": validation_epoch_elapsed
    })

print(f"Training time: {format_time(time.time() - train_start_time)} (HH:MM:SS)")   

Let's convert the statistics array to a dataframe. We'll be able to fetch the stats faster by specifying their columns.

In [ ]:
dataframe_stats = pd.DataFrame(training_stats)
dataframe_stats = dataframe_stats.set_index("epoch")

dataframe_stats

In [ ]:
plt.plot(dataframe_stats["train loss"], "g-o", label="train loss")
plt.plot(dataframe_stats["validation loss"], "b-o", label="validation loss")

plt.title("Training and Validation loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.xticks([1, 2, 3])
plt.legend()
plt.show()

# Predicting

The final part - prediction. It's much simpler than everything we've seen above. It'll be easy to understand, considering you've came this far.

Since our prediction data is in a dataloader (meaning the data is retrieved in batches), we'll `extend` the predictions array instead of `append`ing to it.

Putting logits on the CPU is necessary since calculating the prediction on the GPU would raise an error.

In [ ]:
predictions = []

model.eval()
for batch_input_ids, batch_attention_masks in test_dataloader:
    batch_input_ids = batch_input_ids.to(device)
    batch_attention_masks = batch_attention_masks.to(device)

    with torch.inference_mode():
        outputs = model(
            batch_input_ids,
            attention_mask=batch_attention_masks,
            token_type_ids=None
        )
        
    logits = outputs.logits.detach().cpu()
    prediction = np.argmax(logits, axis=-1).numpy()

    predictions.extend(prediction)

predictions[:10]

Sample submissions are actually very useful. You can just swap the target column values with the predictions array (`list` or `np.ndarray`) and turn the dataframe to a CSV file. I've learned this way too late. Better later than never though!

In [ ]:
submission = pd.read_csv(sample_submission_dir)
submission.sample(5)

In [ ]:
submission["target"] = predictions
submission.sample(5)

And finally, there it is - our submission file.

In [ ]:
submission.to_csv("submission.csv", index=False)

Thanks for coming by. It motivates to see *actual* people taking interest in my work. I'll try to update this every now and then to bring the future explorers a better experience. You can contribute to that by proposing ideas in the comments, which, of course, isn't necessary!

See ya later ;)